# SeRel-LightFM: Kết hợp đặc trưng nội dung, văn bản và đồ thị tri thức

## Tóm tắt (Abstract)
Notebook này triển khai SeRel-LightFM, một biến thể LightFM mở rộng bằng cách đưa đồng thời đặc trưng nội dung, embedding văn bản và embedding đồ thị tri thức (Knowledge Graph Embedding, KGE) vào không gian đặc trưng. Đồ thị tri thức được mở rộng bằng nút người dùng để TransE học biểu diễn cho cả user và item. Các embedding sau đó được gom cụm bằng KMeans và đưa vào LightFM dưới dạng đặc trưng rời rạc có trọng số.

## 1. Giới thiệu (Introduction)
Mục tiêu của phương pháp là cải thiện khả năng gợi ý trong bối cảnh dữ liệu thưa và item cold-start bằng cách kết hợp ba nguồn tín hiệu: metadata có cấu trúc, ngữ nghĩa văn bản và quan hệ đồ thị tri thức. Thiết lập Leave-One-Out được giữ thống nhất với các baseline để bảo đảm so sánh thực nghiệm công bằng.

| Thành phần | Mô tả |
|---|---|
| Mô hình xếp hạng | LightFM với WARP loss |
| Biểu diễn văn bản | Sentence Transformer và KMeans text clusters |
| Biểu diễn tri thức | TransE trên KG mở rộng có user nodes |
| Đặc trưng cuối | Tier 1 item features, Tier 2 text/KGE clusters, Tier 3 user features |


## 2. Phương pháp nghiên cứu (Methodology)

### 2.1 Thiết lập môi trường và cấu hình
Cell cấu hình khai báo nguồn dữ liệu, checkpoint, ngưỡng phản hồi ẩn $\tau$, siêu tham số LightFM và các tham số của từng tầng đặc trưng. Các lựa chọn này được giữ nhất quán với baseline để bảo đảm so sánh thực nghiệm công bằng.


In [ ]:
# [Giải thích cell]
# - Mục đích: nạp thư viện và khai báo toàn bộ cấu hình thực nghiệm của notebook.
# - Đầu vào chính: URL dữ liệu, thư mục checkpoint, ngưỡng phản hồi ẩn và siêu tham số mô hình.
# - Kết quả tạo ra: các hằng số cấu hình được các cell phía sau sử dụng thống nhất.
# - Lưu ý: cấu hình thuộc biến thể SeRel-LightFM đầy đủ, do đó cần giữ cố định khi so sánh với notebook khác.

import os, pickle, ast, warnings
from collections import Counter

import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Nguồn dữ liệu ──────────────────────────────────────────────────────────────
MOVIES_URL  = "https://raw.githubusercontent.com/lynchblue/movie-rating-dataset/main/data/movies.csv"
RATINGS_URL = "https://raw.githubusercontent.com/lynchblue/movie-rating-dataset/main/data/ratings.csv"
USERS_URL   = "https://raw.githubusercontent.com/lynchblue/movie-rating-dataset/main/data/user_profiles.csv"

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Cấu hình phản hồi ẩn ─────────────────────────────────────────────────────────
POSITIVE_THRESHOLD = 7
MIN_RATINGS        = 10

# ── Siêu tham số LightFM (giống Baseline 100% để so sánh công bằng) ──────────
NO_COMPONENTS  = 64
LEARNING_RATE  = 0.05
ITEM_ALPHA     = 1e-6
USER_ALPHA     = 1e-6
MAX_EPOCHS     = 30
PATIENCE       = 5
NUM_THREADS    = 4
K_VALUES       = (5, 10, 20, 50)

# ── Tầng 2: mã hóa văn bản ────────────────────────────────────────────────────
TEXT_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
K_TEXT_CLUSTERS = 200
TEXT_TOP_K      = 3
TEXT_TEMP       = 0.1

# ── Tầng 2: KGE phía item ─────────────────────────────────────────────────
KGE_DIM           = 64          # TransE embedding dim (real-valued)
KGE_EPOCHS        = 100
K_KG_CLUSTERS     = 150         # cluster item KGE embeddings
KG_TOP_K          = 3
KG_TEMP           = 0.1

# ── Tầng 2: KGE phía user ─────────────────────────────────────────
# 481 users → K=30 gives ~16 users/cluster (đủ dense để có pattern)
K_USER_KG_CLUSTERS = 30         # cluster user KGE embeddings
USER_KG_TOP_K      = 3          # mỗi user gán top-K cluster gần nhất
USER_KG_TEMP       = 0.1        # softmax temperature

# ── Cờ điều khiển ablation ────────────────────────────────────────────────────────────
USE_TEXT_CLUSTERS    = True
USE_KG_CLUSTERS      = True
USE_USER_KG_CLUSTERS = True     # MỚI: bật/tắt user KGE cluster features

# ── Quan hệ user-item trong đồ thị tri thức ────────────────────────────────────────────────────
USE_USER_WATCHED_REL = True     # True: thêm cả user_watched; False: chỉ user_likes

# ── GPU Cấu hình thực nghiệm ─────────────────────────────────────────────────────────
import torch

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE_STR = "cuda" if torch.cuda.is_available() else "cpu"

# Batch sizes: GPU cho phép batch lớn hơn nhiều → encode nhanh hơn ~10x
TEXT_BATCH_SIZE = 512 if DEVICE.type == "cuda" else 64   # SentenceTransformer
KGE_BATCH_SIZE  = 4096 if DEVICE.type == "cuda" else 2048 # TransE training

print(f"[KGE-UserNode] Cấu hình: threshold={POSITIVE_THRESHOLD}, components={NO_COMPONENTS}, "
      f"lr={LEARNING_RATE}, patience={PATIENCE}")
print(f"[KGE-UserNode] Split : Leave-One-Out (LOO) per user")
print(f"[KGE-UserNode] Tier 2 Text   : {TEXT_MODEL_NAME} → KMeans({K_TEXT_CLUSTERS}) → soft top-{TEXT_TOP_K}")
print(f"[KGE-UserNode] Tier 2 Item KG: TransE({KGE_DIM}-D) → KMeans({K_KG_CLUSTERS}) → soft top-{KG_TOP_K}")
print(f"[KGE-UserNode] Tier 2 User KG: TransE({KGE_DIM}-D) → KMeans({K_USER_KG_CLUSTERS}) → soft top-{USER_KG_TOP_K}  ← MỚI")
print(f"[KGE-UserNode] Tier 3 User   : binary flags + hour/acc_year/weekday (giống Baseline)")
print()
print(f"[GPU] Device    : {DEVICE_STR.upper()}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    free, total = torch.cuda.mem_get_info(0)
    print(f"[GPU] Name      : {props.name}")
    print(f"[GPU] Memory    : {props.total_memory / 1e9:.1f} GB total  |  {free / 1e9:.1f} GB free")
    print(f"[GPU] SM count  : {props.multi_processor_count}")
    print(f"[GPU] Text batch: {TEXT_BATCH_SIZE}  |  KGE batch: {KGE_BATCH_SIZE}")
else:
    print(f"[GPU] ⚠ CUDA không khả dụng — chạy trên CPU (chậm hơn ~10x)")
    print(f"      Kiểm tra: Runtime → Change runtime type → GPU (Kaggle/Colab)")


[KGE-UserNode] Config: threshold=7, components=64, lr=0.05, patience=5
[KGE-UserNode] Split : Leave-One-Out (LOO) per user
[KGE-UserNode] Tier 2 Text   : paraphrase-multilingual-MiniLM-L12-v2 → KMeans(200) → soft top-3
[KGE-UserNode] Tier 2 Item KG: TransE(64-D) → KMeans(150) → soft top-3
[KGE-UserNode] Tier 2 User KG: TransE(64-D) → KMeans(30) → soft top-3  ← MỚI
[KGE-UserNode] Tier 3 User   : binary flags + hour/acc_year/weekday (giống Baseline)

[GPU] Device    : CUDA
[GPU] Name      : Tesla T4
[GPU] Memory    : 15.6 GB total  |  15.5 GB free
[GPU] SM count  : 40
[GPU] Text batch: 512  |  KGE batch: 4096


### 2.2 Các hàm hỗ trợ
Nhóm hàm hỗ trợ thực hiện chuẩn hóa văn bản, token hóa, lọc token hiếm và xử lý giá trị thiếu. Đây là phần tiền xử lý dùng chung cho toàn bộ pipeline.


In [ ]:
# [Giải thích cell]
# - Mục đích: định nghĩa các hàm tiền xử lý văn bản và token dùng lại trong notebook.
# - Đầu vào chính: các cột metadata dạng chuỗi, thường chứa token phân tách bằng dấu `|`.
# - Kết quả tạo ra: chuỗi đã làm sạch, danh sách token và tập token đủ phổ biến để làm đặc trưng.
# - Lưu ý: các hàm này chỉ chuẩn hóa dữ liệu; không học tham số từ validation hoặc test.

def cleanup_column(series):
    """Fix encoding artefacts in pipe-separated string Series."""
    replacements = {
        '| ': '|', '\xc3\xa9': 'é', '\xc3\x81': 'Á', '\xc3\x93': 'Ó',
        '\xc3\xa1': 'á', '\xc3\xb3': 'ó', '\xc3\xb1': 'ñ', '\xc3\xad': 'í',
        '\xc3\xba': 'ú', 'Ã©': 'é', 'Ã¡': 'á', 'Ã³': 'ó', 'Ã±': 'ñ',
        'Ãº': 'ú',
    }
    col = series.copy()
    for bad, good in replacements.items():
        col = col.str.replace(bad, good, regex=False)
    return col


def tokenize(series, sep='|'):
    """Split pipe-separated Series → list-of-lists. NaN → []."""
    return [
        [t.strip() for t in s.split(sep) if t.strip()]
        if isinstance(s, str) else []
        for s in series
    ]


def filter_rare(token_lists, min_count):
    """Loại token xuất hiện < min_count lần."""
    if min_count <= 1:
        return token_lists
    counter = Counter()
    for toks in token_lists:
        counter.update(set(toks))
    keep = {t for t, c in counter.items() if c >= min_count}
    return [[t for t in toks if t in keep] for toks in token_lists]


def fix_plot(p):
    """Fix byte-string encoded plots và encoding artifacts."""
    if not isinstance(p, str) or p.strip() in ('', 'nan'):
        return ""
    if p.startswith("b'") or p.startswith('b"'):
        try:
            decoded = ast.literal_eval(p)
            if isinstance(decoded, bytes):
                return decoded.decode('utf-8', errors='replace')
        except Exception:
            try:
                decoded = ast.literal_eval(p)
                if isinstance(decoded, bytes):
                    return decoded.decode('latin-1', errors='replace')
            except Exception:
                return p[2:-1]
    return p


print("Helpers defined: cleanup_column, tokenize, filter_rare, fix_plot")

Helpers defined: cleanup_column, tokenize, filter_rare, fix_plot


### 2.3 Nạp dữ liệu
Các bảng `movies.csv`, `ratings.csv` và `user_profiles.csv` được nạp và chuẩn hóa kiểu thời gian. Dữ liệu rating cung cấp tín hiệu tương tác, trong khi metadata phim và hồ sơ người dùng cung cấp đặc trưng phụ.


In [ ]:
# [Giải thích cell]
# - Mục đích: nạp ba bảng dữ liệu gốc gồm rating, hồ sơ người dùng và metadata phim.
# - Đầu vào chính: các đường dẫn `RATINGS_URL`, `USERS_URL` và `MOVIES_URL`.
# - Kết quả tạo ra: `ratings_df`, `users_df`, `movies_df` và cột thời gian đã được chuẩn hóa.
# - Lưu ý: kiểm tra kích thước bảng sau khi nạp giúp phát hiện sớm lỗi đọc dữ liệu.

ratings_df = pd.read_csv(RATINGS_URL)
users_df   = pd.read_csv(USERS_URL)
movies_df  = pd.read_csv(MOVIES_URL)

ratings_df["date"] = pd.to_datetime(ratings_df["date"])

print(f"Users   : {users_df['id'].nunique()}")
print(f"Movies  : {movies_df['id'].nunique()}")
print(f"Ratings : {len(ratings_df):,}")
print(f"Rating range: {ratings_df['rate'].min()} – {ratings_df['rate'].max()}")
print(f"Date range  : {ratings_df['date'].min().date()} – {ratings_df['date'].max().date()}")

Users   : 482
Movies  : 78628
Ratings : 1,172,038
Rating range: 1 – 10
Date range  : 2002-08-14 – 2021-04-01


### 2.4 Lọc dữ liệu hợp lệ
Các rating không có hồ sơ người dùng, người dùng có quá ít rating và movie không xuất hiện trong metadata được loại bỏ. Bước này bảo đảm tập dữ liệu còn lại phù hợp với Leave-One-Out và xây dựng đặc trưng.


In [ ]:
# [Giải thích cell]
# - Mục đích: thực hiện một bước xử lý trong pipeline thực nghiệm của notebook.
# - Đầu vào chính: các biến đã được tạo ở những cell trước.
# - Kết quả tạo ra: biến trung gian hoặc bảng kết quả dùng cho các bước tiếp theo.
# - Lưu ý: comment này được bổ sung để giúp người đọc theo dõi luồng xử lý mà không cần chạy lại code.

# 2.1 Lọc orphan users
users_in_profile = set(users_df["id"].unique())
ratings_df = ratings_df[ratings_df["id"].isin(users_in_profile)].copy()
print(f"Sau lọc orphan users    : {len(ratings_df):,} ratings")

# 2.2 Lọc user có ít hơn MIN_RATINGS
rating_counts = ratings_df.groupby("id")["movie_id"].count()
valid_users   = rating_counts[rating_counts >= MIN_RATINGS].index
ratings_df    = ratings_df[ratings_df["id"].isin(valid_users)].copy()
users_df      = users_df[users_df["id"].isin(valid_users)].copy()
print(f"Sau lọc min {MIN_RATINGS} ratings : {len(ratings_df):,} ratings, {ratings_df['id'].nunique()} users")

# 2.3 Lọc movie không tồn tại
movies_in_table = set(movies_df["id"].unique())
ratings_df = ratings_df[ratings_df["movie_id"].isin(movies_in_table)].copy()
print(f"Sau lọc missing movies  : {len(ratings_df):,} ratings")

Sau lọc orphan users    : 963,266 ratings
Sau lọc min 10 ratings : 963,231 ratings, 474 users
Sau lọc missing movies  : 963,228 ratings


### 2.5 Chuẩn hóa văn bản metadata
Các trường văn bản phân tách bằng dấu `|` và trường `plot` được làm sạch để giảm lỗi encoding. Bước này tạo đầu vào ổn định cho token hóa và mã hóa văn bản.


In [ ]:
# [Giải thích cell]
# - Mục đích: làm sạch lỗi encoding trong metadata phim trước khi token hóa hoặc hiển thị.
# - Đầu vào chính: các cột văn bản như `genres`, `topics`, `actors`, `directors` và `plot`.
# - Kết quả tạo ra: metadata nhất quán hơn, giảm nhiễu cho đặc trưng nội dung và text encoder.
# - Lưu ý: đây là xử lý thuộc tính item, không dùng nhãn validation/test để học mô hình.

# 3.1 Fix encoding artifacts
for col in ["country_name", "genres", "topics", "directors", "actors",
            "script", "producer", "music", "photography"]:
    movies_df[col] = cleanup_column(movies_df[col])

# 3.2 Fix byte-string plots
movies_df["plot_clean"] = movies_df["plot"].apply(fix_plot)

print("Text cleanup hoàn tất.")
print(f"Plot NaN/empty: {(movies_df['plot_clean'] == '').sum():,} / {len(movies_df):,}")

Text cleanup hoàn tất.
Plot NaN/empty: 2,442 / 78,628


### 2.6 Chia dữ liệu Leave-One-Out và áp dụng threshold
Quy trình dữ liệu được tách thành hai giai đoạn rõ ràng:

1. **Chia Leave-One-Out trên rating gốc:** `ratings_df` sau khi lọc hợp lệ vẫn giữ đầy đủ giá trị rating. Với mỗi người dùng, rating được sắp xếp theo thời gian; rating mới nhất được đưa vào `test_df`, rating liền trước được đưa vào `valid_df`, và toàn bộ rating còn lại được đưa vào `train_df`.
2. **Áp dụng threshold sau khi đã chia:** các split `train_df`, `valid_df` và `test_df` vẫn là dữ liệu rating gốc. Khi xây dựng tương tác phản hồi ẩn, chỉ các dòng thỏa $r_{u,i} \geq \tau$ mới được chuyển thành tương tác dương.

Điều này có nghĩa là threshold không được dùng để quyết định dòng nào thuộc train, validation hay test. Threshold chỉ quyết định dòng nào trở thành tương tác dương khi huấn luyện hoặc đánh giá. Cách làm này giữ đúng thứ tự thời gian của hành vi người dùng và tránh làm thay đổi tương tác mới nhất trước khi chia dữ liệu.


In [ ]:
# [Giải thích cell]
# - Mục đích: chia dữ liệu Leave-One-Out trên rating gốc, trước khi áp dụng threshold phản hồi ẩn.
# - Cách làm: sắp xếp rating theo thời gian cho từng user; lấy rating mới nhất làm test, rating liền trước làm validation, phần còn lại làm train.
# - Kết quả tạo ra: `train_df`, `valid_df`, `test_df`; các bảng này vẫn giữ rating gốc, gồm cả rating cao và thấp.
# - Lưu ý: threshold `rate >= POSITIVE_THRESHOLD` chỉ được áp dụng ở các bước xây dựng interaction/evaluation sau đó.

ratings_df = ratings_df.sort_values(["id", "date"]).reset_index(drop=True)

train_idx, valid_idx, test_idx = [], [], []

for uid, group in ratings_df.groupby("id", sort=False):
    idx = group.index.tolist()
    if len(idx) < 3:
        train_idx.extend(idx)
        continue
    test_idx.append(idx[-1])
    valid_idx.append(idx[-2])
    train_idx.extend(idx[:-2])

train_df = ratings_df.loc[train_idx].copy().reset_index(drop=True)
valid_df = ratings_df.loc[valid_idx].copy().reset_index(drop=True)
test_df  = ratings_df.loc[test_idx].copy().reset_index(drop=True)

print(f"Train : {len(train_df):>7,} ratings  |  {train_df['id'].nunique()} users")
print(f"Valid : {len(valid_df):>7,} ratings  |  {valid_df['id'].nunique()} users  (1 rating/user)")
print(f"Test  : {len(test_df):>7,} ratings  |  {test_df['id'].nunique()} users  (1 rating/user)")

Train : 962,280 ratings  |  474 users
Valid :     474 ratings  |  474 users  (1 rating/user)
Test  :     474 ratings  |  474 users  (1 rating/user)


### 2.7 Phân loại item warm/cold
Item warm đã xuất hiện trong train, còn item cold chưa xuất hiện trong train. Phân tách này giúp đánh giá riêng khả năng tổng quát hóa của mô hình đối với item thiếu lịch sử tương tác.


In [ ]:
# [Giải thích cell]
# - Mục đích: tách item trong tập kiểm thử thành warm và cold theo sự xuất hiện trong train.
# - Đầu vào chính: `train_df` và `test_df`.
# - Kết quả tạo ra: `test_warm_df`, `test_cold_df` cùng các phiên bản positive nếu có.
# - Lưu ý: phân tách này giúp đánh giá riêng khả năng xử lý item cold-start.

movies_in_train = set(train_df["movie_id"].unique())

test_warm_mask = test_df["movie_id"].isin(movies_in_train)
test_cold_mask = ~test_warm_mask

test_warm_df = test_df[test_warm_mask].copy().reset_index(drop=True)
test_cold_df = test_df[test_cold_mask].copy().reset_index(drop=True)

print(f"Unique items in train : {len(movies_in_train):,}")
print(f"\nTEST total : {len(test_df):>6,} ratings")
print(f"TEST warm  : {len(test_warm_df):>6,} ratings  "
      f"({len(test_warm_df)/len(test_df)*100:.1f}%)")
print(f"TEST cold  : {len(test_cold_df):>6,} ratings  "
      f"({len(test_cold_df)/len(test_df)*100:.1f}%)")

Unique items in train : 75,115

TEST total :    474 ratings
TEST warm  :    446 ratings  (94.1%)
TEST cold  :     28 ratings  (5.9%)


### 2.8 Xây dựng Tier 1 item features
Tier 1 gồm các đặc trưng nội dung có cấu trúc như `genre`, `topic`, `country`, `year` và `duration`. Biến liên tục được chuẩn hóa trước khi đưa vào từ điển đặc trưng item.


In [ ]:
# [Giải thích cell]
# - Mục đích: xây dựng đặc trưng item có cấu trúc từ metadata phim.
# - Đầu vào chính: thể loại, chủ đề, quốc gia, năm phát hành và thời lượng.
# - Kết quả tạo ra: `item_feat_dicts`, trong đó mỗi movie_id ánh xạ tới các feature có trọng số.
# - Lưu ý: biến liên tục được chuẩn hóa để tránh lấn át đặc trưng phân loại.

# Token hóa các cột phân loại
genre_toks   = tokenize(movies_df["genres"])
topic_toks   = tokenize(movies_df["topics"])
country_toks = filter_rare(tokenize(movies_df["country_name"]), min_count=2)

# Tham số chuẩn hóa biến liên tục
year_min   = movies_df["year_published"].min()
year_range = movies_df["year_published"].max() - year_min
dur_cap    = movies_df["duration"].quantile(0.99)

# Xây dựng từ điển đặc trưng item: {movie_id: {feature_name: weight}}
item_feat_dicts = {}
for i in range(len(movies_df)):
    row = movies_df.iloc[i]
    mid = int(row["id"])
    feats = {}

    if pd.notna(row["year_published"]) and year_range > 0:
        feats["year"] = (row["year_published"] - year_min) / year_range
    if pd.notna(row["duration"]) and dur_cap > 0:
        feats["duration"] = min(row["duration"] / dur_cap, 1.0)

    for g in genre_toks[i]:
        feats[f"genre:{g}"] = 1.0
    for t in topic_toks[i]:
        feats[f"topic:{t}"] = 1.0
    for c in country_toks[i]:
        feats[f"country:{c}"] = 1.0

    item_feat_dicts[mid] = feats

# Tóm tắt kiểm tra
all_tier1_tags = set()
for feats in item_feat_dicts.values():
    all_tier1_tags.update(feats.keys())

print(f"Tier 1 features: {len(all_tier1_tags)} unique tags")
print(f"  genre/topic/country/continuous → giữ nguyên 100% baseline")

Tier 1 features: 590 unique tags
  genre/topic/country/continuous → giữ nguyên 100% baseline


### 2.9 Xây dựng Tier 3 user features
Tier 3 gồm các đặc trưng hồ sơ tĩnh của người dùng. Việc giữ lại nhóm đặc trưng này giúp LightFM khai thác side information ở phía user ngoài tín hiệu tương tác.


In [ ]:
# [Giải thích cell]
# - Mục đích: chuyển hồ sơ người dùng thành đặc trưng rời rạc cho LightFM.
# - Đầu vào chính: các cột hành vi và hồ sơ tĩnh trong `users_df`.
# - Kết quả tạo ra: `user_feat_dicts`, ánh xạ user_id tới tập feature mô tả người dùng.
# - Lưu ý: đặc trưng user không lấy từ validation/test, giúp giảm nguy cơ rò rỉ dữ liệu.

def bin_preferred_hour(h):
    if pd.isna(h):       return "hour:unknown"
    if 5 <= h <= 11:     return "hour:morning"
    elif 12 <= h <= 17:  return "hour:afternoon"
    elif 18 <= h <= 22:  return "hour:evening"
    else:                return "hour:night"

def bin_account_year(y):
    if pd.isna(y):     return "acc:unknown"
    if y < 2010:       return "acc:pre2010"
    elif y <= 2013:    return "acc:2010_2013"
    else:              return "acc:post2013"

users_df["hour_bin"]     = users_df["preferred_hour"].apply(bin_preferred_hour)
users_df["acc_year_bin"] = users_df["account_creation_year"].apply(bin_account_year)
users_df["weekday_bin"]  = "weekday:" + users_df["preferred_weekday"].astype(str)

BINARY_FLAGS = ["night_owl", "early_bird", "weekend_tweeter", "week_tweeter", "geo_enabled"]

user_feat_dicts = {}
for _, row in users_df.iterrows():
    uid   = int(row["id"])
    feats = {}

    for col in BINARY_FLAGS:
        if row.get(col, 0) == 1:
            feats[f"user:{col}"] = 1.0

    feats[row["hour_bin"]]     = 1.0
    feats[row["acc_year_bin"]] = 1.0
    feats[row["weekday_bin"]]  = 1.0

    user_feat_dicts[uid] = feats

all_user_tags = set()
for feats in user_feat_dicts.values():
    all_user_tags.update(feats.keys())

print(f"Tier 3 user features: {len(all_user_tags)} unique tags")
print(f"  Users với features: {len(user_feat_dicts)} / {users_df['id'].nunique()}")

Tier 3 user features: 19 unique tags
  Users với features: 474 / 474


### 2.10 Mã hóa văn bản bằng Sentence Transformer
Text Encoder không sử dụng `train_df`, `valid_df` hoặc `test_df`. Thay vào đó, cell này mã hóa metadata của toàn bộ catalogue phim trong `movies_df` sau khi đã được làm sạch văn bản.

Đầu vào của Text Encoder gồm các trường như tiêu đề, thể loại, chủ đề, đạo diễn, diễn viên và `plot_clean`. Vì đây là thuộc tính nội dung tĩnh của item, bước mã hóa văn bản không phụ thuộc vào rating và không áp dụng threshold $r_{u,i} \geq \tau$.

Kết quả của bước này là `text_embeddings`, tức embedding ngữ nghĩa cho từng phim trong `movies_df`. Các embedding này sau đó được gom cụm bằng KMeans để tạo `text_cluster:*` feature cho LightFM.


In [ ]:
# [Giải thích cell]
# - Mục đích: cài các thư viện cần thiết để mã hóa văn bản và gom cụm embedding.
# - Lưu ý: cell này chỉ chuẩn bị môi trường; không tạo dữ liệu huấn luyện hay kết quả đánh giá.

pip install sentence-transformers scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# [Giải thích cell]
# - Mục đích: mã hóa metadata văn bản của toàn bộ `movies_df`, không phải chỉ phim trong `train_df`.
# - Đầu vào chính: title, genres, topics, directors, actors và `plot_clean` đã làm sạch.
# - Kết quả tạo ra: `text_embeddings`, mỗi hàng tương ứng với một phim trong `movies_df`.
# - Lưu ý: bước này không dùng rating, không dùng threshold và không dùng nhãn validation/test.

from sentence_transformers import SentenceTransformer

# ── Bước 1: xây dựng đầu vào văn bản đa trường và bỏ qua trường rỗng ──
def build_text_input(row):
    parts = []
    title = row.get("original_title")
    if isinstance(title, str) and title.strip():
        parts.append(title.strip())

    genres = row.get("genres")
    if isinstance(genres, str) and genres.strip():
        parts.append(f"Generos: {genres.strip()}")

    topics = row.get("topics")
    if isinstance(topics, str) and topics.strip() and topics.strip().lower() != "nan":
        parts.append(f"Temas: {topics.strip()}")

    directors = row.get("directors")
    if isinstance(directors, str) and directors.strip():
        parts.append(f"Directores: {directors.strip()}")

    actors = row.get("actors")
    if isinstance(actors, str) and actors.strip():
        actor_list = [a.strip() for a in actors.split("|") if a.strip()][:5]
        if actor_list:
            parts.append(f"Reparto: {', '.join(actor_list)}")

    plot = row.get("plot_clean")
    if isinstance(plot, str) and plot.strip():
        parts.append(plot.strip())

    return ". ".join(parts) if parts else "(no metadata)"

texts = [build_text_input(movies_df.iloc[i]) for i in range(len(movies_df))]
print(f"Built {len(texts):,} text inputs")
print(f"Sample (first movie, truncated): {texts[0][:200]}...")
print(f"Text length stats: mean={np.mean([len(t) for t in texts]):.0f} chars, "
      f"median={np.median([len(t) for t in texts]):.0f} chars")

# ── Bước 2: nạp mô hình và mã hóa văn bản trên GPU ────────────────────────────────────
print(f"\nLoading {TEXT_MODEL_NAME} trên {DEVICE_STR.upper()}...")
st_model = SentenceTransformer(TEXT_MODEL_NAME, device=DEVICE_STR)

if DEVICE.type == "cuda":
    # Giải phóng bộ nhớ đệm trước khi mã hóa để tối ưu VRAM
    torch.cuda.empty_cache()
    print(f"[GPU] VRAM trước encode: "
          f"{torch.cuda.memory_allocated(0)/1e6:.0f} MB allocated / "
          f"{torch.cuda.memory_reserved(0)/1e6:.0f} MB reserved")

text_embeddings = st_model.encode(
    texts,
    batch_size=TEXT_BATCH_SIZE,       # 512 trên GPU, 64 trên CPU
    show_progress_bar=True,
    normalize_embeddings=True,        # L2-norm → cosine sim = dot product
    convert_to_numpy=True,
    device=DEVICE_STR,
)

if DEVICE.type == "cuda":
    print(f"[GPU] VRAM sau encode : "
          f"{torch.cuda.memory_allocated(0)/1e6:.0f} MB allocated")

print(f"\nText embeddings shape : {text_embeddings.shape}")
print(f"Mean L2 norm          : {np.linalg.norm(text_embeddings, axis=1).mean():.4f}")

del st_model
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    print(f"[GPU] VRAM sau del model: {torch.cuda.memory_allocated(0)/1e6:.0f} MB")


Built 78,628 text inputs
Sample (first movie, truncated): Chaos Theory. Generos: Drama|Comedia. Directores: Marcos Siega. Reparto: Ryan Reynolds, Emily Mortimer, Stuart Townsend, Sarah Chalke, Mike Erwin. Frank (Ryan Reynolds), famoso autor de un best seller...
Text length stats: mean=519 chars, median=487 chars

Loading paraphrase-multilingual-MiniLM-L12-v2 trên CUDA...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[GPU] VRAM trước encode: 471 MB allocated / 484 MB reserved


Batches:   0%|          | 0/154 [00:00<?, ?it/s]

[GPU] VRAM sau encode : 480 MB allocated

Text embeddings shape : (78628, 384)
Mean L2 norm          : 1.0000
[GPU] VRAM sau del model: 480 MB


### 2.11 Gom cụm embedding văn bản
Embedding văn bản được phân cụm bằng KMeans với $K=200$. Với mỗi item, các cụm gần nhất được gán trọng số bằng softmax trên cosine similarity:

$$
w_c = \frac{\exp(s_c / T)}{\sum_{c'} \exp(s_{c'} / T)},
$$

trong đó $s_c$ là độ tương đồng với centroid $c$ và $T$ là temperature.


In [ ]:
# [Giải thích cell]
# - Mục đích: gom cụm embedding văn bản để biến vector dense thành feature rời rạc cho LightFM.
# - Đầu vào chính: `text_embeddings` đã chuẩn hóa.
# - Kết quả tạo ra: `text_cluster_assignments`, gồm các cụm gần nhất và trọng số của từng phim.
# - Lưu ý: softmax theo temperature giúp phân phối trọng số mượt hơn giữa các cụm gần nhau.

from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize as sk_normalize

text_clusters = None
text_cluster_assignments = None  # {movie_idx: [(cluster_id, weight), ...]}

if USE_TEXT_CLUSTERS and text_embeddings is not None:
    print(f"Running KMeans (K={K_TEXT_CLUSTERS}) on text embeddings...")
    kmeans_text = KMeans(
        n_clusters=K_TEXT_CLUSTERS,
        random_state=42,
        n_init=10,
        verbose=0,
    )
    kmeans_text.fit(text_embeddings)

    # L2-normalize centroids để cosine sim = dot product
    centroids_text = sk_normalize(kmeans_text.cluster_centers_, norm="l2", axis=1)

    # Cosine similarity giữa từng movie embedding và mọi centroid
    # text_embeddings đã L2-normalized, centroids cũng vậy → dot product = cosine
    sims_text = text_embeddings @ centroids_text.T  # (n_movies, K_TEXT_CLUSTERS)

    # Lấy top-K cluster cho từng movie
    top_idx = np.argpartition(-sims_text, TEXT_TOP_K, axis=1)[:, :TEXT_TOP_K]

    # Sắp xếp các cụm gần nhất theo từng item
    text_cluster_assignments = []
    for i in range(len(movies_df)):
        clusters = top_idx[i]
        cluster_sims = sims_text[i, clusters]
        order = np.argsort(-cluster_sims)
        clusters_sorted = clusters[order]
        sims_sorted = cluster_sims[order]

        # Tính trọng số softmax có điều chỉnh temperature
        exp_sims = np.exp(sims_sorted / TEXT_TEMP)
        weights = exp_sims / exp_sims.sum()

        text_cluster_assignments.append([
            (int(c), float(w)) for c, w in zip(clusters_sorted, weights)
        ])

    # Kiểm tra chẩn đoán
    cluster_sizes = np.bincount(top_idx[:, 0], minlength=K_TEXT_CLUSTERS)
    print(f"  Top-1 cluster size: min={cluster_sizes.min()}, "
          f"max={cluster_sizes.max()}, mean={cluster_sizes.mean():.1f}")
    print(f"  Sample movie 0 top-{TEXT_TOP_K} text clusters: {text_cluster_assignments[0]}")
    print(f"  Sample movie 100 top-{TEXT_TOP_K} text clusters: {text_cluster_assignments[100]}")
else:
    print("Text clustering: SKIPPED (USE_TEXT_CLUSTERS=False)")

Running KMeans (K=200) on text embeddings...
  Top-1 cluster size: min=80, max=840, mean=393.1
  Sample movie 0 top-3 text clusters: [(88, 0.37377652525901794), (30, 0.3185960054397583), (24, 0.30762746930122375)]
  Sample movie 100 top-3 text clusters: [(93, 0.3551696240901947), (103, 0.3312114179134369), (180, 0.31361904740333557)]


### 2.12 Xây dựng đồ thị tri thức mở rộng và huấn luyện TransE
Đồ thị tri thức sử dụng hai nhóm dữ liệu khác nhau, trong đó cần phân biệt rõ thời điểm áp dụng threshold:

- **Item-attribute triples:** được tạo từ metadata của toàn bộ `movies_df`, gồm các quan hệ như `has_genre`, `directed_by`, `acted_by`, `from_country` và `has_topic`. Nhóm quan hệ này không dùng rating, không dùng `train_df`/`valid_df`/`test_df` và không phụ thuộc vào threshold.
- **User-item triples:** chỉ được tạo từ `train_df` sau khi chia LOO. `train_df` ở thời điểm này vẫn là dataset rating gốc, chưa bị lọc bỏ rating thấp. Threshold $r_{u,i} \geq \tau$ chỉ được dùng để phân loại cạnh trong KG: rating dương tạo `user_likes`, còn rating thấp hơn ngưỡng có thể tạo `user_watched` nếu `USE_USER_WATCHED_REL=True`.

Do đó, KGE **không dùng dataset sau khi lọc threshold theo nghĩa chỉ còn positive interactions**. KGE dùng train split trước khi lọc threshold, rồi dùng threshold để gán loại quan hệ user-item. `valid_df` và `test_df` của bài toán gợi ý không được dùng để tạo cạnh user-item trong KG. Sau khi gộp triples, PyKEEN tự chia nội bộ KG thành training/testing triples để huấn luyện TransE; split này khác với `valid_df`, `test_df` của recommendation.


In [ ]:
# [Giải thích cell]
# - Mục đích: cài PyKEEN và PyTorch để huấn luyện mô hình TransE trên đồ thị tri thức.
# - Lưu ý: bước cài đặt thư viện được tách riêng để dễ tái lập môi trường thực nghiệm.

pip install pykeen torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 1.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# [Giải thích cell]
# - Mục đích: xây dựng KG từ metadata phim và tương tác user-item trong train để huấn luyện TransE.
# - Item-attribute triples: dùng toàn bộ `movies_df` sau làm sạch; phần này không phụ thuộc rating hay threshold.
# - User-item triples: dùng `train_df` sau LOO nhưng trước khi lọc bỏ rating thấp; `valid_df`/`test_df` không được dùng.
# - Threshold trong KGE: `rate >= POSITIVE_THRESHOLD` tạo `user_likes`; rating thấp hơn ngưỡng có thể tạo `user_watched` nếu `USE_USER_WATCHED_REL=True`.
# - Lưu ý: KGE không dùng dataset đã lọc chỉ còn positive interactions; threshold chỉ phân loại quan hệ trong train split.

import torch
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline as kge_pipeline

# ══════════════════════════════════════════════════════════════════════════════
# Bước 1: xây dựng bộ ba item-attribute (9 relations, giống Proposed-Final)
# ══════════════════════════════════════════════════════════════════════════════
pipe_cols = {
    "genres":       "has_genre",
    "directors":    "directed_by",
    "actors":       "acted_by",
    "country_name": "from_country",
    "topics":       "has_topic",
    "script":       "written_by",
    "producer":     "produced_by",
    "music":        "scored_by",
    "photography":  "shot_by",
}

item_attr_triples = []
for _, row in movies_df.iterrows():
    mid = f"movie:{int(row['id'])}"
    for col, rel in pipe_cols.items():
        val = row.get(col)
        if pd.notna(val) and isinstance(val, str):
            for token in val.split("|"):
                token = token.strip()
                if token and token.lower() != "nan":
                    item_attr_triples.append((mid, rel, f"{col}:{token}"))

print(f"[KG Layer 1] Item-Attribute triples: {len(item_attr_triples):,}  ({len(pipe_cols)} relations)")

# ══════════════════════════════════════════════════════════════════════════════
# Bước 2: xây dựng bộ ba user-item chỉ từ train để tránh rò rỉ dữ liệu
# ══════════════════════════════════════════════════════════════════════════════
user_item_triples = []

# train_df ? ??y l? raw train split sau LOO: v?n g?m c? rating cao v? th?p.
# Threshold KH?NG ???c d?ng ?? lo?i b? to?n b? rating th?p kh?i KGE;
# n? ch? quy?t ??nh c?nh user-item thu?c lo?i user_likes hay user_watched.

# 2a. user_likes: t??ng t?c d??ng trong train (rate >= POSITIVE_THRESHOLD)
positive_train = train_df[train_df["rate"] >= POSITIVE_THRESHOLD]
for _, row in positive_train.iterrows():
    u_key = f"user:{int(row['id'])}"
    m_key = f"movie:{int(row['movie_id'])}"
    user_item_triples.append((u_key, "user_likes", m_key))

print(f"[KG Layer 2] user_likes triples   : {len(user_item_triples):,}  "
      f"({positive_train['id'].nunique()} users → {positive_train['movie_id'].nunique()} movies)")

# 2b. user_watched: tất cả interactions (nếu flag bật)
n_watched = 0
if USE_USER_WATCHED_REL:
    negative_train = train_df[train_df["rate"] < POSITIVE_THRESHOLD]
    for _, row in negative_train.iterrows():
        u_key = f"user:{int(row['id'])}"
        m_key = f"movie:{int(row['movie_id'])}"
        user_item_triples.append((u_key, "user_watched", m_key))
    n_watched = len(negative_train)
    print(f"[KG Layer 2] user_watched triples : {n_watched:,}  "
          f"(negative/neutral interactions from train)")

print(f"[KG Layer 2] Total User-Item triples: {len(user_item_triples):,}")

# ══════════════════════════════════════════════════════════════════════════════
# Bước 3: gộp bộ ba và thống kê
# ══════════════════════════════════════════════════════════════════════════════
all_triples = item_attr_triples + user_item_triples
triples_arr = np.array(all_triples)

unique_entities  = set(triples_arr[:, 0]) | set(triples_arr[:, 2])
unique_relations = set(triples_arr[:, 1])
n_user_entities  = sum(1 for e in unique_entities if e.startswith("user:"))
n_movie_entities = sum(1 for e in unique_entities if e.startswith("movie:"))
n_attr_entities  = len(unique_entities) - n_user_entities - n_movie_entities

print(f"\n[KG Merged]")
print(f"  Total triples  : {len(all_triples):,}")
print(f"  Total relations: {len(unique_relations)}  {sorted(unique_relations)}")
print(f"  Total entities : {len(unique_entities):,}")
print(f"    ├─ user  entities: {n_user_entities:,}")
print(f"    ├─ movie entities: {n_movie_entities:,}")
print(f"    └─ attr  entities: {n_attr_entities:,}")

# ══════════════════════════════════════════════════════════════════════════════
# Bước 4: xây dựng TriplesFactory và chia tập
# ══════════════════════════════════════════════════════════════════════════════
tf = TriplesFactory.from_labeled_triples(triples_arr)
training_tf, testing_tf = tf.split([0.9, 0.1], random_state=42)
print(f"\n[KG Split] train {training_tf.num_triples:,} / test {testing_tf.num_triples:,}")
print(f"[KG Split] Entities in factory: {training_tf.num_entities:,}")


[KG Layer 1] Item-Attribute triples: 1,539,858  (9 relations)
[KG Layer 2] user_likes triples   : 412,438  (473 users → 36478 movies)
[KG Layer 2] user_watched triples : 549,842  (negative/neutral interactions from train)
[KG Layer 2] Total User-Item triples: 962,280

[KG Merged]
  Total triples  : 2,502,138
  Total relations: 11  [np.str_('acted_by'), np.str_('directed_by'), np.str_('from_country'), np.str_('has_genre'), np.str_('has_topic'), np.str_('produced_by'), np.str_('scored_by'), np.str_('shot_by'), np.str_('user_likes'), np.str_('user_watched'), np.str_('written_by')]
  Total entities : 493,364
    ├─ user  entities: 474
    ├─ movie entities: 78,628
    └─ attr  entities: 414,262

[KG Split] train 2,249,652 / test 249,962
[KG Split] Entities in factory: 493,364


In [ ]:
# [Giải thích cell]
# - Mục đích: huấn luyện TransE trên các triples của KG đã xây dựng.
# - Đầu vào chính: `training_tf` và `testing_tf` do PyKEEN tách từ `all_triples`.
# - Kết quả tạo ra: mô hình TransE chứa embedding cho user, movie và attribute entity.
# - Lưu ý: KG split của PyKEEN khác với `train_df`/`valid_df`/`test_df` trong bài toán gợi ý.

# ══════════════════════════════════════════════════════════════════════════════
# Bước 5: huấn luyện TransE trên KG hợp nhất (user + item + attribute entities)
# ══════════════════════════════════════════════════════════════════════════════
# PyKEEN tự detect GPU qua tham số `device`. Khi CUDA khả dụng:
#   - Model parameters được đặt trên GPU
#   - Batch sampling và forward pass chạy trên GPU
#   - Tốc độ train nhanh hơn CPU ~5–15x tùy GPU

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info(0)[0] / 1e9
    print(f"[GPU] VRAM khả dụng trước TransE: {free_gb:.1f} GB")

print(f"[TransE] Device   : {DEVICE_STR.upper()}")
print(f"[TransE] Entities : {training_tf.num_entities:,}  (users + movies + attributes)")
print(f"[TransE] Relations: {training_tf.num_relations}")
print(f"[TransE] Triples  : {training_tf.num_triples:,}")
print(f"[TransE] Dim      : {KGE_DIM}-D real-valued")
print(f"[TransE] Epochs   : {KGE_EPOCHS}  |  Batch: {KGE_BATCH_SIZE}")

result = kge_pipeline(
    training=training_tf,
    testing=testing_tf,
    model="TransE",
    model_kwargs=dict(embedding_dim=KGE_DIM),
    training_kwargs=dict(
        num_epochs=KGE_EPOCHS,
        batch_size=KGE_BATCH_SIZE,    # 4096 trên GPU, 2048 trên CPU
    ),
    optimizer="Adam",
    optimizer_kwargs=dict(lr=1e-3),
    random_seed=42,
    device=DEVICE_STR,               # ← GPU/CPU dispatch cho PyKEEN
)

print(f"\n[TransE] Training complete.")
print(f"[TransE] Model device: {next(result.model.parameters()).device}")
print(f"[TransE] Entity embedding matrix: {training_tf.num_entities} × {KGE_DIM}")

if DEVICE.type == "cuda":
    used_gb = torch.cuda.memory_allocated(0) / 1e9
    peak_gb = torch.cuda.max_memory_allocated(0) / 1e9
    print(f"[GPU] VRAM sau training: {used_gb:.2f} GB used | {peak_gb:.2f} GB peak")


[GPU] VRAM khả dụng trước TransE: 15.4 GB
[TransE] Device   : CUDA
[TransE] Entities : 493,364  (users + movies + attributes)
[TransE] Relations: 11
[TransE] Triples  : 2,249,652
[TransE] Dim      : 64-D real-valued
[TransE] Epochs   : 100  |  Batch: 4096


Training epochs on cuda:0:   0%|          | 0/100 [00:00<?, ?epoch/s]

Training batches on cuda:0:   0%|          | 0.00/550 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/550 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/550 [00:00<?, ?batch/s]

...

Training batches on cuda:0:   0%|          | 0.00/550 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/550 [00:00<?, ?batch/s]

Evaluating on cuda:0:   0%|          | 0.00/250k [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 1385.85s seconds



[TransE] Training complete.
[TransE] Model device: cuda:0
[TransE] Entity embedding matrix: 493364 × 64
[GPU] VRAM sau training: 0.57 GB used | 12.71 GB peak


In [ ]:
# [Giải thích cell]
# - Mục đích: trích xuất embedding TransE cho movie và user entity sau khi huấn luyện KG.
# - Đầu vào chính: mô hình TransE và ánh xạ entity trong `training_tf`.
# - Kết quả tạo ra: `kg_embeddings` cho toàn bộ `movies_df` và `user_kg_matrix_normed` cho toàn bộ `users_df`.
# - Lưu ý: entity không xuất hiện trong KG sẽ nhận vector 0 trước khi chuẩn hóa an toàn.

# ══════════════════════════════════════════════════════════════════════════════
# Mục 9c + 9d - Trích xuất embedding cho toàn bộ entity trong một batch (GPU-efficient)
# ══════════════════════════════════════════════════════════════════════════════
# Thay vì extract từng entity một (78K+ lần forward pass), ta:
#   1. Chạy 1 lần forward pass cho ALL entity IDs trên GPU
#   2. Map kết quả về numpy rồi index theo movie/user
# → Nhanh hơn hàng chục lần so với per-entity loop

entity_to_id = training_tf.entity_to_id
n_entities   = training_tf.num_entities

# Đảm bảo model ở đúng device (PyKEEN thường đã làm điều này)
result.model = result.model.to(DEVICE)
result.model.eval()

# ── Trích xuất hàng loạt toàn bộ entity embeddings ───────────────────────────────
print(f"[9c/9d] Extracting {n_entities:,} entity embeddings trên {DEVICE_STR.upper()}...")

EXTRACT_CHUNK = 8192  # số entities mỗi chunk (tránh OOM trên GPU nhỏ)
all_embs_list = []

with torch.no_grad():
    for start in range(0, n_entities, EXTRACT_CHUNK):
        end     = min(start + EXTRACT_CHUNK, n_entities)
        ids_gpu = torch.arange(start, end, device=DEVICE)
        chunk   = result.model.entity_representations[0](ids_gpu)
        all_embs_list.append(chunk.cpu().float().numpy())

all_entity_embs = np.concatenate(all_embs_list, axis=0)  # (n_entities, KGE_DIM)
kg_out_dim = all_entity_embs.shape[1]
print(f"[9c/9d] All entity embeddings: {all_entity_embs.shape}  (dtype={all_entity_embs.dtype})")

# ── Mục 9c: Map về ITEM embeddings ──────────────────────────────────────────
print("\n[9c] Mapping → item embeddings...")
kg_embeddings = np.zeros((len(movies_df), kg_out_dim), dtype=np.float32)
item_found = 0
for i, mid in enumerate(movies_df["id"]):
    key = f"movie:{int(mid)}"
    if key in entity_to_id:
        kg_embeddings[i] = all_entity_embs[entity_to_id[key]]
        item_found += 1

# L2 normalize
norms = np.linalg.norm(kg_embeddings, axis=1, keepdims=True)
norms = np.where(norms == 0, 1.0, norms)
kg_embeddings = kg_embeddings / norms

print(f"[9c] Item embeddings: {kg_embeddings.shape}  | matched {item_found:,}/{len(movies_df):,} movies")
print(f"[9c] Mean L2 norm (item): {np.linalg.norm(kg_embeddings, axis=1).mean():.4f}")

# ── Mục 9d: Map về USER embeddings ──────────────────────────────────────────
print("\n[9d] Mapping → user embeddings...")
user_ids_ordered = users_df["id"].tolist()
user_kg_matrix   = np.zeros((len(user_ids_ordered), kg_out_dim), dtype=np.float32)
user_found       = 0
users_not_in_kg  = []

for i, uid in enumerate(user_ids_ordered):
    key = f"user:{int(uid)}"
    if key in entity_to_id:
        user_kg_matrix[i] = all_entity_embs[entity_to_id[key]]
        user_found += 1
    else:
        users_not_in_kg.append(uid)

# L2 normalize
norms = np.linalg.norm(user_kg_matrix, axis=1, keepdims=True)
norms = np.where(norms == 0, 1.0, norms)
user_kg_matrix_normed = user_kg_matrix / norms

user_has_kg_emb = np.linalg.norm(user_kg_matrix, axis=1) > 1e-6

print(f"[9d] User embeddings : {user_kg_matrix.shape}  | matched {user_found}/{len(user_ids_ordered)} users")
print(f"[9d] Users NOT in KG : {len(users_not_in_kg)} "
      f"({'none' if not users_not_in_kg else str(users_not_in_kg[:5])})")
if user_has_kg_emb.any():
    print(f"[9d] Mean L2 norm (user): "
          f"{np.linalg.norm(user_kg_matrix_normed[user_has_kg_emb], axis=1).mean():.4f}")

# Phân tích: user vs item center cosine similarity
item_center  = kg_embeddings.mean(axis=0)
user_centers = user_kg_matrix_normed[user_has_kg_emb].mean(axis=0)
cos_sim = float(np.dot(item_center, user_centers) /
                (np.linalg.norm(item_center) * np.linalg.norm(user_centers) + 1e-8))
print(f"[9d] Cosine sim (user-center vs item-center): {cos_sim:.4f} "
      f"(>0 → users & items cùng phía trong KGE space)")

# ── Giải phóng bộ nhớ GPU ───────────────────────────────────────────────────────
del all_embs_list, all_entity_embs, result
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    print(f"\n[GPU] VRAM sau cleanup: {torch.cuda.memory_allocated(0)/1e6:.0f} MB allocated")


[9c/9d] Extracting 493,364 entity embeddings trên CUDA...
[9c/9d] All entity embeddings: (493364, 64)  (dtype=float32)

[9c] Mapping → item embeddings...
[9c] Item embeddings: (78628, 64)  | matched 78,628/78,628 movies
[9c] Mean L2 norm (item): 1.0000

[9d] Mapping → user embeddings...
[9d] User embeddings : (474, 64)  | matched 474/474 users
[9d] Users NOT in KG : 0 (none)
[9d] Mean L2 norm (user): 1.0000
[9d] Cosine sim (user-center vs item-center): -0.6979 (>0 → users & items cùng phía trong KGE space)

[GPU] VRAM sau cleanup: 569 MB allocated


### 2.13 Gom cụm embedding KGE của item và user
Embedding KGE của item và user được chuẩn hóa rồi phân cụm bằng KMeans. Cụm KGE được dùng như đặc trưng rời rạc để đưa tín hiệu cấu trúc đồ thị vào LightFM.


In [ ]:
# [Giải thích cell]
# - Mục đích: gom cụm embedding KGE để biến vector đồ thị thành feature rời rạc cho LightFM.
# - Đầu vào item: `kg_embeddings` theo thứ tự toàn bộ `movies_df`.
# - Đầu vào user: `user_kg_matrix_normed` theo thứ tự toàn bộ `users_df`.
# - Kết quả tạo ra: `kg_cluster_assignments` cho item và `user_kg_cluster_assignments` cho user.

from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize as sk_normalize

# ══════════════════════════════════════════════════════════════════════════════
# Mục 9e - KMeans Clustering trên ITEM KGE embeddings (giống Proposed-Final)
# ══════════════════════════════════════════════════════════════════════════════
kg_cluster_assignments = None  # list of [(cluster_id, weight), ...] per item

if USE_KG_CLUSTERS and kg_embeddings is not None:
    print(f"[9e] KMeans (K={K_KG_CLUSTERS}) trên {len(kg_embeddings):,} item KGE embeddings...")
    kmeans_kg = KMeans(n_clusters=K_KG_CLUSTERS, random_state=42, n_init=10, verbose=0)
    kmeans_kg.fit(kg_embeddings)

    centroids_kg = sk_normalize(kmeans_kg.cluster_centers_, norm="l2", axis=1)
    sims_kg = kg_embeddings @ centroids_kg.T  # (n_movies, K_KG_CLUSTERS)

    top_idx = np.argpartition(-sims_kg, KG_TOP_K, axis=1)[:, :KG_TOP_K]

    kg_cluster_assignments = []
    for i in range(len(movies_df)):
        clusters     = top_idx[i]
        cluster_sims = sims_kg[i, clusters]
        order        = np.argsort(-cluster_sims)
        clusters_s   = clusters[order]
        sims_s       = cluster_sims[order]
        exp_s        = np.exp(sims_s / KG_TEMP)
        weights      = exp_s / exp_s.sum()
        kg_cluster_assignments.append([(int(c), float(w)) for c, w in zip(clusters_s, weights)])

    sizes = np.bincount(top_idx[:, 0], minlength=K_KG_CLUSTERS)
    print(f"[9e] Item KGE cluster sizes: min={sizes.min()}, max={sizes.max()}, mean={sizes.mean():.1f}")
    print(f"[9e] Sample movie 0  → top-{KG_TOP_K}: {kg_cluster_assignments[0]}")
    print(f"[9e] Sample movie 100 → top-{KG_TOP_K}: {kg_cluster_assignments[100]}")
else:
    print("[9e] Item KG clustering: SKIPPED (USE_KG_CLUSTERS=False)")

# ══════════════════════════════════════════════════════════════════════════════
# Mục 9f - KMeans Clustering trên USER KGE embeddings — MỚI
# ══════════════════════════════════════════════════════════════════════════════
user_kg_cluster_assignments = None  # dict: uid → [(cluster_id, weight), ...]

if USE_USER_KG_CLUSTERS and user_kg_matrix_normed is not None:
    n_users_total = len(user_ids_ordered)
    n_users_with_emb = int(user_has_kg_emb.sum())
    print(f"\n[9f] KMeans (K={K_USER_KG_CLUSTERS}) trên {n_users_total} user KGE embeddings "
          f"({n_users_with_emb} với embedding thực, {n_users_total - n_users_with_emb} zero)...")

    # Fit trên tất cả users (kể cả zero vectors — họ sẽ được gán cluster gần nhất)
    kmeans_user = KMeans(
        n_clusters=K_USER_KG_CLUSTERS,
        random_state=42,
        n_init=10,
        verbose=0,
    )
    kmeans_user.fit(user_kg_matrix_normed)

    centroids_user = sk_normalize(kmeans_user.cluster_centers_, norm="l2", axis=1)
    sims_user = user_kg_matrix_normed @ centroids_user.T  # (n_users, K_USER_KG_CLUSTERS)

    top_idx_user = np.argpartition(-sims_user, USER_KG_TOP_K, axis=1)[:, :USER_KG_TOP_K]

    user_kg_cluster_assignments = {}
    for i, uid in enumerate(user_ids_ordered):
        clusters     = top_idx_user[i]
        cluster_sims = sims_user[i, clusters]
        order        = np.argsort(-cluster_sims)
        clusters_s   = clusters[order]
        sims_s       = cluster_sims[order]
        exp_s        = np.exp(sims_s / USER_KG_TEMP)
        weights      = exp_s / exp_s.sum()
        user_kg_cluster_assignments[uid] = [(int(c), float(w)) for c, w in zip(clusters_s, weights)]

    u_sizes = np.bincount(top_idx_user[:, 0], minlength=K_USER_KG_CLUSTERS)
    print(f"[9f] User KGE cluster sizes: min={u_sizes.min()}, max={u_sizes.max()}, mean={u_sizes.mean():.1f}")

    # Phân tích cluster purity: user trong cùng cluster có sở thích tương tự?
    # Proxy: đo intra-cluster cosine similarity trung bình
    label_user = kmeans_user.labels_
    intra_sims = []
    for c in range(K_USER_KG_CLUSTERS):
        mask = label_user == c
        if mask.sum() < 2:
            continue
        cluster_embs = user_kg_matrix_normed[mask]
        gram = cluster_embs @ cluster_embs.T
        n = len(cluster_embs)
        intra_sims.append((gram.sum() - n) / (n * (n - 1)))  # mean off-diagonal
    print(f"[9f] Mean intra-cluster cosine sim: {np.mean(intra_sims):.4f}  "
          f"(higher → users trong cùng cluster prefer phim tương tự)")

    # In mẫu kết quả gán cụm
    sample_uid = user_ids_ordered[0]
    print(f"[9f] Sample user {sample_uid} → top-{USER_KG_TOP_K}: {user_kg_cluster_assignments[sample_uid]}")
    if len(user_ids_ordered) > 5:
        sample_uid2 = user_ids_ordered[5]
        print(f"[9f] Sample user {sample_uid2} → top-{USER_KG_TOP_K}: {user_kg_cluster_assignments[sample_uid2]}")
else:
    print("[9f] User KG clustering: SKIPPED (USE_USER_KG_CLUSTERS=False)")


[9e] KMeans (K=150) trên 78,628 item KGE embeddings...
[9e] Item KGE cluster sizes: min=344, max=1045, mean=524.2
[9e] Sample movie 0  → top-3: [(132, 0.3505115509033203), (93, 0.3294013440608978), (95, 0.3200870156288147)]
[9e] Sample movie 100 → top-3: [(26, 0.3932874798774719), (60, 0.3340921103954315), (21, 0.27262043952941895)]

[9f] KMeans (K=30) trên 474 user KGE embeddings (474 với embedding thực, 0 zero)...
[9f] User KGE cluster sizes: min=1, max=32, mean=15.8
[9f] Mean intra-cluster cosine sim: 0.6328  (higher → users trong cùng cluster prefer phim tương tự)
[9f] Sample user 103007 → top-3: [(12, 0.46617284417152405), (14, 0.2838312089443207), (11, 0.24999597668647766)]
[9f] Sample user 1087549 → top-3: [(7, 0.6132368445396423), (14, 0.20846880972385406), (22, 0.17829427123069763)]


### 2.14 Tích hợp đặc trưng Tier 2
Các cụm text, item KGE và user KGE được đưa vào `item_feat_dicts` hoặc `user_feat_dicts` với trọng số đã hiệu chỉnh. Bảng ánh xạ dưới đây mô tả nguồn đặc trưng và vị trí tích hợp trong LightFM.


In [ ]:
# [Giải thích cell]
# - Mục đích: đưa các feature Tier 2 vào từ điển đặc trưng item/user trước khi fit LightFM.
# - Đầu vào chính: text clusters, KGE clusters và các cờ ablation tương ứng.
# - Kết quả tạo ra: `item_feat_dicts` và/hoặc `user_feat_dicts` đã có thêm feature embedding dạng cụm.
# - Lưu ý: trọng số feature được nhân scale để cân bằng ảnh hưởng giữa các nhóm đặc trưng.

# ══════════════════════════════════════════════════════════════════════════════
# Inject ITEM Tier 2 features (giống Proposed-Final)
# ══════════════════════════════════════════════════════════════════════════════

# 1. Text cluster features → item_feat_dicts
n_text_added = 0
if text_cluster_assignments is not None:
    for i in range(len(movies_df)):
        mid = int(movies_df.iloc[i]["id"])
        for cluster_id, weight in text_cluster_assignments[i]:
            item_feat_dicts[mid][f"text_cluster:{cluster_id}"] = weight
    n_text_added = K_TEXT_CLUSTERS

# 2. Item KGE cluster features → item_feat_dicts
n_kg_added = 0
if kg_cluster_assignments is not None:
    for i in range(len(movies_df)):
        mid = int(movies_df.iloc[i]["id"])
        for cluster_id, weight in kg_cluster_assignments[i]:
            item_feat_dicts[mid][f"kg_cluster:{cluster_id}"] = weight
    n_kg_added = K_KG_CLUSTERS

# ══════════════════════════════════════════════════════════════════════════════
# Inject USER Tier 2 features — MỚI
# ══════════════════════════════════════════════════════════════════════════════

# 3. User KGE cluster features → user_feat_dicts
n_user_kg_added = 0
if user_kg_cluster_assignments is not None:
    missing_users = 0
    for uid, cluster_list in user_kg_cluster_assignments.items():
        if uid not in user_feat_dicts:
            # User có KGE embedding nhưng không có Tier 3 features → tạo mới
            user_feat_dicts[uid] = {}
            missing_users += 1
        for cluster_id, weight in cluster_list:
            user_feat_dicts[uid][f"user_kg_cluster:{cluster_id}"] = weight
    n_user_kg_added = K_USER_KG_CLUSTERS
    if missing_users > 0:
        print(f"[Inject] Cảnh báo: {missing_users} users có KGE cluster nhưng thiếu Tier 3 → initialized empty dict")

# ══════════════════════════════════════════════════════════════════════════════
# Tóm tắt kiểm tra
# ══════════════════════════════════════════════════════════════════════════════

# Refresh tag sets
all_item_tags = set()
for feats in item_feat_dicts.values():
    all_item_tags.update(feats.keys())

all_user_tags = set()
for feats in user_feat_dicts.values():
    all_user_tags.update(feats.keys())

n_text_tags    = len({t for t in all_item_tags if t.startswith("text_cluster:")})
n_kg_tags      = len({t for t in all_item_tags if t.startswith("kg_cluster:")})
n_user_kg_tags = len({t for t in all_user_tags if t.startswith("user_kg_cluster:")})
n_tier3_tags   = len({t for t in all_user_tags if not t.startswith("user_kg_cluster:")})

print("=" * 65)
print("Feature Space Tóm tắt kiểm tra")
print("=" * 65)
print(f"ITEM features: {len(all_item_tags)} total")
print(f"  ├─ Tier 1 (categorical + continuous): {len(all_tier1_tags)}")
print(f"  ├─ Tier 2 text clusters             : {n_text_tags}")
print(f"  └─ Tier 2 item KGE clusters         : {n_kg_tags}")
print()
print(f"USER features: {len(all_user_tags)} total")
print(f"  ├─ Tier 3 (Twitter behavioral)      : {n_tier3_tags}")
print(f"  └─ Tier 2 user KGE clusters (MỚI)   : {n_user_kg_tags}")
print()

# Sample verification
sample_mid = list(item_feat_dicts.keys())[0]
sf = item_feat_dicts[sample_mid]
print(f"Sample item (mid={sample_mid}):")
print(f"  Tier1 : {[(k,v) for k,v in sf.items() if not k.startswith(('text_cluster:','kg_cluster:'))][:4]}")
print(f"  Text  : {[(k,v) for k,v in sf.items() if k.startswith('text_cluster:')]}")
print(f"  KG    : {[(k,v) for k,v in sf.items() if k.startswith('kg_cluster:')]}")
print()
sample_uid = list(user_feat_dicts.keys())[0]
uf = user_feat_dicts[sample_uid]
print(f"Sample user (uid={sample_uid}):")
print(f"  Tier3     : {[(k,v) for k,v in uf.items() if not k.startswith('user_kg_cluster:')]}")
print(f"  User KGE  : {[(k,v) for k,v in uf.items() if k.startswith('user_kg_cluster:')]}")


Feature Space Summary
ITEM features: 940 total
  ├─ Tier 1 (categorical + continuous): 590
  ├─ Tier 2 text clusters             : 200
  └─ Tier 2 item KGE clusters         : 150

USER features: 49 total
  ├─ Tier 3 (Twitter behavioral)      : 19
  └─ Tier 2 user KGE clusters (MỚI)   : 30

Sample item (mid=100052):
  Tier1 : [('year', np.float64(0.8986486486486487)), ('duration', np.float64(0.4777777777777778)), ('genre:Drama', 1.0), ('genre:Comedia', 1.0)]
  Text  : [('text_cluster:88', 0.37377652525901794), ('text_cluster:30', 0.3185960054397583), ('text_cluster:24', 0.30762746930122375)]
  KG    : [('kg_cluster:132', 0.3505115509033203), ('kg_cluster:93', 0.3294013440608978), ('kg_cluster:95', 0.3200870156288147)]

Sample user (uid=103007):
  Tier3     : [('user:night_owl', 1.0), ('user:geo_enabled', 1.0), ('hour:evening', 1.0), ('acc:pre2010', 1.0), ('weekday:1', 1.0)]
  User KGE  : [('user_kg_cluster:12', 0.46617284417152405), ('user_kg_cluster:14', 0.2838312089443207), ('user_kg_

### 2.15 Lưu checkpoint sau tiền xử lý và Tier 2
Checkpoint lưu lại đặc trưng item/user, embedding, cụm KMeans và cấu hình thực nghiệm. Bước này hỗ trợ tái lập mà không cần chạy lại Sentence Transformer, TransE và KMeans.


In [ ]:
# [Giải thích cell]
# - Mục đích: lưu các artifact sau tiền xử lý để tái sử dụng ở các lần chạy sau.
# - Nội dung lưu: raw split sau LOO (`train_df`, `valid_df`, `test_df`) và các feature/embedding đã tạo.
# - Với notebook có KGE, checkpoint lưu KGE clusters đã được tạo từ KG dùng metadata `movies_df` và user-item edges từ raw `train_df`.
# - Lưu ý: checkpoint không tự chạy lại threshold; threshold đã được lưu trong cấu hình để các bước LightFM/evaluation dùng nhất quán.

PROPOSED_CHECKPOINT = os.path.join(CHECKPOINT_DIR, "kge_usernode_preprocessed.pkl")

artifacts = {
    "ratings_df":    ratings_df,
    "users_df":      users_df,
    "movies_df":     movies_df,
    "train_df":      train_df,
    "valid_df":      valid_df,
    "test_df":       test_df,
    "test_warm_df":  test_warm_df,
    "test_cold_df":  test_cold_df,
    # Đặc trưng item
    "item_feat_dicts":          item_feat_dicts,
    "all_item_tags":            all_item_tags,
    "all_tier1_tags":           all_tier1_tags,
    "text_cluster_assignments": text_cluster_assignments,
    "kg_cluster_assignments":   kg_cluster_assignments,
    # Đặc trưng user
    "user_feat_dicts":               user_feat_dicts,
    "all_user_tags":                  all_user_tags,
    "user_kg_cluster_assignments":    user_kg_cluster_assignments,  # MỚI
    "user_ids_ordered":               user_ids_ordered,              # MỚI
    # Cấu hình
    "config": {
        "POSITIVE_THRESHOLD": POSITIVE_THRESHOLD, "MIN_RATINGS": MIN_RATINGS,
        "NO_COMPONENTS": NO_COMPONENTS, "LEARNING_RATE": LEARNING_RATE,
        "ITEM_ALPHA": ITEM_ALPHA, "USER_ALPHA": USER_ALPHA,
        "MAX_EPOCHS": MAX_EPOCHS, "PATIENCE": PATIENCE,
        "NUM_THREADS": NUM_THREADS, "K_VALUES": K_VALUES,
        # Text
        "K_TEXT_CLUSTERS": K_TEXT_CLUSTERS, "TEXT_TOP_K": TEXT_TOP_K, "TEXT_TEMP": TEXT_TEMP,
        # Item KGE
        "KGE_DIM": KGE_DIM, "K_KG_CLUSTERS": K_KG_CLUSTERS, "KG_TOP_K": KG_TOP_K, "KG_TEMP": KG_TEMP,
        # User KGE (MỚI)
        "K_USER_KG_CLUSTERS": K_USER_KG_CLUSTERS, "USER_KG_TOP_K": USER_KG_TOP_K,
        "USER_KG_TEMP": USER_KG_TEMP,
        # Flags
        "USE_TEXT_CLUSTERS": USE_TEXT_CLUSTERS, "USE_KG_CLUSTERS": USE_KG_CLUSTERS,
        "USE_USER_KG_CLUSTERS": USE_USER_KG_CLUSTERS,
        "USE_USER_WATCHED_REL": USE_USER_WATCHED_REL,
    },
}

with open(PROPOSED_CHECKPOINT, "wb") as f:
    pickle.dump(artifacts, f, protocol=pickle.HIGHEST_PROTOCOL)

size_mb = os.path.getsize(PROPOSED_CHECKPOINT) / (1024 * 1024)
print(f"[Checkpoint] Saved: {PROPOSED_CHECKPOINT}  ({size_mb:.1f} MB)")
print(f"[Checkpoint] Artifacts: {list(artifacts.keys())}")


[Checkpoint] Saved: /kaggle/working/checkpoints/kge_usernode_preprocessed.pkl  (165.3 MB)
[Checkpoint] Artifacts: ['ratings_df', 'users_df', 'movies_df', 'train_df', 'valid_df', 'test_df', 'test_warm_df', 'test_cold_df', 'item_feat_dicts', 'all_item_tags', 'all_tier1_tags', 'text_cluster_assignments', 'kg_cluster_assignments', 'user_feat_dicts', 'all_user_tags', 'user_kg_cluster_assignments', 'user_ids_ordered', 'config']


### 2.16 Nạp checkpoint
Cell này nạp lại checkpoint tiền xử lý và khôi phục các biến cần thiết cho bước khởi tạo `Dataset`. Quy trình nạp checkpoint giữ nguyên đặc trưng đã được tạo trước đó.


In [ ]:
# [Giải thích cell]
# - Mục đích: nạp lại checkpoint đã lưu để tiếp tục từ bước xây dựng ma trận LightFM.
# - Đầu vào chính: file `.pkl` chứa dữ liệu và đặc trưng đã tiền xử lý.
# - Kết quả tạo ra: các biến notebook được khôi phục về trạng thái sau preprocessing.
# - Lưu ý: cần bảo đảm checkpoint khớp với đúng biến thể thực nghiệm đang chạy.

import os, pickle, ast, warnings
from collections import Counter
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

# ── Mục 10c - Load checkpoint ────────────────────────────────────────────────
LOAD_FROM_CHECKPOINT = True

if LOAD_FROM_CHECKPOINT:
    checkpoint_name = "kge_usernode_preprocessed.pkl"
    checkpoint_candidates = [
        os.path.join(".", checkpoint_name),
        checkpoint_name,
        os.path.join("/content", checkpoint_name),
        os.path.join("/kaggle/working/checkpoints", checkpoint_name),
    ]
    checkpoint_path = next((p for p in checkpoint_candidates if os.path.exists(p)), None)
    if checkpoint_path is None:
        raise FileNotFoundError(
            f"Không tìm thấy checkpoint '{checkpoint_name}'. "
            "Vui lòng chạy các Mục 0–10 trước hoặc upload file checkpoint."
        )

    with open(checkpoint_path, "rb") as f:
        artifacts = pickle.load(f)

    # ── Restore dataframes ───────────────────────────────────────────────────
    ratings_df   = artifacts["ratings_df"]
    users_df     = artifacts["users_df"]
    movies_df    = artifacts["movies_df"]
    train_df     = artifacts["train_df"]
    valid_df     = artifacts["valid_df"]
    test_df      = artifacts["test_df"]
    test_warm_df = artifacts["test_warm_df"]
    test_cold_df = artifacts["test_cold_df"]

    # ── Restore item features ────────────────────────────────────────────────
    item_feat_dicts          = artifacts["item_feat_dicts"]
    all_item_tags            = artifacts["all_item_tags"]
    all_tier1_tags           = artifacts["all_tier1_tags"]
    text_cluster_assignments = artifacts.get("text_cluster_assignments")
    kg_cluster_assignments   = artifacts.get("kg_cluster_assignments")

    # ── Restore user features ────────────────────────────────────────────────
    user_feat_dicts              = artifacts["user_feat_dicts"]
    all_user_tags                = artifacts["all_user_tags"]
    user_kg_cluster_assignments  = artifacts.get("user_kg_cluster_assignments")  # MỚI
    user_ids_ordered             = artifacts.get("user_ids_ordered", users_df["id"].tolist())

    # ── Restore config ────────────────────────────────────────────────────────
    cfg = artifacts.get("config", {})
    POSITIVE_THRESHOLD   = cfg.get("POSITIVE_THRESHOLD", 7)
    MIN_RATINGS          = cfg.get("MIN_RATINGS", 10)
    NO_COMPONENTS        = cfg.get("NO_COMPONENTS", 64)
    LEARNING_RATE        = cfg.get("LEARNING_RATE", 0.05)
    ITEM_ALPHA           = cfg.get("ITEM_ALPHA", 1e-6)
    USER_ALPHA           = cfg.get("USER_ALPHA", 1e-6)
    MAX_EPOCHS           = cfg.get("MAX_EPOCHS", 30)
    PATIENCE             = cfg.get("PATIENCE", 5)
    NUM_THREADS          = cfg.get("NUM_THREADS", 4)
    K_VALUES             = tuple(cfg.get("K_VALUES", (5, 10, 20, 50)))
    K_TEXT_CLUSTERS      = cfg.get("K_TEXT_CLUSTERS", 200)
    TEXT_TOP_K           = cfg.get("TEXT_TOP_K", 3)
    TEXT_TEMP            = cfg.get("TEXT_TEMP", 0.1)
    KGE_DIM              = cfg.get("KGE_DIM", 64)
    K_KG_CLUSTERS        = cfg.get("K_KG_CLUSTERS", 150)
    KG_TOP_K             = cfg.get("KG_TOP_K", 3)
    KG_TEMP              = cfg.get("KG_TEMP", 0.1)
    K_USER_KG_CLUSTERS   = cfg.get("K_USER_KG_CLUSTERS", 30)   # MỚI
    USER_KG_TOP_K        = cfg.get("USER_KG_TOP_K", 3)         # MỚI
    USER_KG_TEMP         = cfg.get("USER_KG_TEMP", 0.1)        # MỚI
    USE_TEXT_CLUSTERS    = cfg.get("USE_TEXT_CLUSTERS", True)
    USE_KG_CLUSTERS      = cfg.get("USE_KG_CLUSTERS", True)
    USE_USER_KG_CLUSTERS = cfg.get("USE_USER_KG_CLUSTERS", True)  # MỚI

    # Tóm tắt kiểm tra
    n_user_kg_tags = len({t for t in all_user_tags if t.startswith("user_kg_cluster:")})
    print(f"[Load] Checkpoint: {checkpoint_path}")
    print(f"[Load] Train/Valid/Test: {len(train_df):,} / {len(valid_df):,} / {len(test_df):,}")
    print(f"[Load] Item feature tags: {len(all_item_tags):,}")
    print(f"[Load] User feature tags: {len(all_user_tags):,}  "
          f"(Tier3: {len(all_user_tags)-n_user_kg_tags}, UserKGE: {n_user_kg_tags})")
    print(f"[Load] Cấu hình: components={NO_COMPONENTS}, lr={LEARNING_RATE}, epochs={MAX_EPOCHS}")
    print(f"[Load] user_kg_cluster_assignments: "
          f"{'loaded' if user_kg_cluster_assignments else 'NOT FOUND (ablation: user KGE disabled)'}")
else:
    print("LOAD_FROM_CHECKPOINT=False → dùng dữ liệu từ các Mục trên.")


[Load] Checkpoint: ./kge_usernode_preprocessed.pkl
[Load] Train/Valid/Test: 962,280 / 474 / 474
[Load] Item feature tags: 940
[Load] User feature tags: 49  (Tier3: 19, UserKGE: 30)
[Load] Config: components=64, lr=0.05, epochs=30
[Load] user_kg_cluster_assignments: loaded


### 2.17 Khởi tạo `Dataset` của LightFM
Đối tượng `Dataset` của LightFM được fit trên toàn bộ catalogue hợp lệ:

- `users=users_df["id"]`: toàn bộ user còn lại sau bước lọc hợp lệ.
- `items=movies_df["id"]`: toàn bộ movie metadata sau bước lọc hợp lệ.
- `user_features` và `item_features`: toàn bộ tên feature đã tạo từ metadata, text clusters và/hoặc KGE clusters.

Bước này chỉ khai báo không gian user, item và feature cho LightFM. Mô hình chưa học từ validation/test tại đây. Việc huấn luyện mô hình chỉ bắt đầu khi gọi `model.fit_partial(...)` với `train_interactions`, vốn được tạo từ các tương tác dương trong `train_df`.


In [ ]:
# [Giải thích cell]
# - Mục đích: cài LightFM, thư viện được dùng để huấn luyện mô hình gợi ý lai.
# - Lưu ý: cell này không huấn luyện mô hình; chỉ bảo đảm package có sẵn trong runtime.

pip install lightfm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 11.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lightfm: filename=lightfm-1.17-cp311-cp311-linux_x86_64.whl size=831129 sha256=a620a8ef032a6e22bf8fa696ad6921b0931d542688775c6092465ba2d874482c
  Stored in directory: /root/.cache/pip/wheels/b9/0d/8a/0729d2e6e3ca2a898ba55201f905da7db3f838a33df5b3fcdd
Successfully built lightfm


In [ ]:
# [Giải thích cell]
# - Mục đích: fit không gian user/item/feature cho LightFM và tạo sparse feature matrices.
# - Đầu vào catalogue: toàn bộ `users_df` và `movies_df` sau lọc hợp lệ.
# - Đầu vào feature: `user_feat_dicts` và `item_feat_dicts`, có thể gồm metadata, text clusters và KGE clusters.
# - Lưu ý: cell này chưa huấn luyện mô hình; huấn luyện chỉ dùng `train_interactions` ở bước sau.

from lightfm import LightFM
from lightfm.data import Dataset

# Refresh tag sets từ dicts hiện tại (bao gồm cả user_kg_cluster nếu đã inject)
all_item_tags = set()
for feats in item_feat_dicts.values():
    all_item_tags.update(feats.keys())

all_user_tags = set()
for feats in user_feat_dicts.values():
    all_user_tags.update(feats.keys())

n_user_kg_tags = len({t for t in all_user_tags if t.startswith("user_kg_cluster:")})
n_tier3_tags   = len(all_user_tags) - n_user_kg_tags

# Fit đối tượng Dataset
dataset = Dataset()
dataset.fit(
    users=users_df["id"].tolist(),
    items=movies_df["id"].tolist(),
    user_features=list(all_user_tags),
    item_features=list(all_item_tags),
)

# Build feature matrices (normalize=False: weights đã được calibrated per Tier)
item_features_raw = [(mid, feats) for mid, feats in item_feat_dicts.items()]
user_features_raw = [(uid, feats) for uid, feats in user_feat_dicts.items()]

item_feature_matrix = dataset.build_item_features(item_features_raw, normalize=False)
user_feature_matrix = dataset.build_user_features(user_features_raw, normalize=False)

print(f"User feature matrix : {user_feature_matrix.shape}")
print(f"  ├─ Tier 3 (Twitter behavioral)  : {n_tier3_tags} features")
print(f"  └─ Tier 2 user KGE clusters      : {n_user_kg_tags} features  ← MỚI")
print(f"Item feature matrix : {item_feature_matrix.shape}")
print(f"  ├─ Tier 1 (categorical+cont)     : {len(all_tier1_tags)} features")
n_text = len({t for t in all_item_tags if t.startswith('text_cluster:')})
n_kg   = len({t for t in all_item_tags if t.startswith('kg_cluster:')})
print(f"  ├─ Tier 2 text clusters          : {n_text} features")
print(f"  └─ Tier 2 item KGE clusters      : {n_kg} features")
print()
print(f"User feature density: {user_feature_matrix.nnz / (user_feature_matrix.shape[0]*user_feature_matrix.shape[1])*100:.4f}%")
print(f"Item feature density: {item_feature_matrix.nnz / (item_feature_matrix.shape[0]*item_feature_matrix.shape[1])*100:.4f}%")


User feature matrix : (474, 523)
  ├─ Tier 3 (Twitter behavioral)  : 19 features
  └─ Tier 2 user KGE clusters      : 30 features  ← MỚI
Item feature matrix : (78628, 79568)
  ├─ Tier 1 (categorical+cont)     : 590 features
  ├─ Tier 2 text clusters          : 200 features
  └─ Tier 2 item KGE clusters      : 150 features

User feature density: 1.6172%
Item feature density: 0.0171%


### 2.18 Xây dựng interactions nhị phân
Các bảng `train_df`, `valid_df` và `test_df` vẫn chứa rating gốc sau khi chia LOO. Tại bước này, threshold mới được áp dụng để tạo ma trận phản hồi ẩn:

$$
R_{u,i} = \mathbb{1}(r_{u,i} \geq \tau).
$$

`train_interactions` được tạo từ các rating dương trong `train_df` và là dữ liệu duy nhất dùng để huấn luyện LightFM. `valid_inter` và `test_inter` chỉ được dùng để early stopping/đánh giá, không được đưa vào `fit_partial`. Khi xếp hạng, các item đã xuất hiện trong raw `train_df` của user được mask để tránh gợi ý lại lịch sử đã biết.


In [ ]:
# [Giải thích cell]
# - Mục đích: áp dụng threshold sau LOO để tạo ma trận phản hồi ẩn cho từng split.
# - Dữ liệu train model: `train_interactions`, chỉ gồm các dòng trong `train_df` có `rate >= POSITIVE_THRESHOLD`.
# - Dữ liệu đánh giá: `valid_inter` và `test_inter`, cũng chỉ giữ held-out rating dương sau threshold.
# - Lưu ý: rating thấp hơn ngưỡng không là ground truth dương; raw `train_df` vẫn được dùng để mask item đã xem.

def build_interactions_binary(df, dataset, threshold=POSITIVE_THRESHOLD):
    positives = df[df["rate"] >= threshold]
    if len(positives) == 0:
        return None
    pairs = [(int(r["id"]), int(r["movie_id"])) for _, r in positives.iterrows()]
    interactions, _ = dataset.build_interactions(pairs)
    return interactions

train_interactions = build_interactions_binary(train_df, dataset)
valid_inter        = build_interactions_binary(valid_df, dataset)
test_inter         = build_interactions_binary(test_df, dataset)
test_warm_inter    = build_interactions_binary(test_warm_df, dataset)
test_cold_inter    = build_interactions_binary(test_cold_df, dataset)

def nnz(m): return m.nnz if m is not None else 0

print(f"Threshold = {POSITIVE_THRESHOLD} (binary implicit, no sample_weight)")
print(f"Train interactions : {nnz(train_interactions):,}")
print(f"Valid interactions : {nnz(valid_inter):,}")
print(f"Test  interactions : {nnz(test_inter):,}")
print(f"  ├─ Warm items    : {nnz(test_warm_inter):,}")
print(f"  └─ Cold items    : {nnz(test_cold_inter):,}")

Threshold = 7 (binary implicit, no sample_weight)
Train interactions : 412,438
Valid interactions : 240
Test  interactions : 262
  ├─ Warm items    : 253
  └─ Cold items    : 9


## 3. Thực nghiệm và Kết quả (Experiments & Results)

### 3.1 Các chỉ số đánh giá Top-$K$
Các chỉ số Precision@$K$, Recall@$K$, NDCG@$K$, HR@$K$ và MRR@$K$ được tính sau khi mask item đã xuất hiện trong train. Cùng một bộ chỉ số được sử dụng cho baseline, ablation và mô hình đề xuất.


In [ ]:
# [Giải thích cell]
# - Mục đích: định nghĩa hàm đánh giá Top-K trên các held-out interactions dương.
# - Đầu vào chính: ma trận held-out sau threshold, `train_interactions` để mask lịch sử và mô hình sinh điểm.
# - Kết quả tạo ra: Precision@K, Recall@K, NDCG@K, HR@K và MRR@K.
# - Lưu ý: validation/test chỉ được dùng để đo hiệu năng; không cập nhật trọng số mô hình.

def evaluate_metrics(model, test_interactions, train_interactions,
                     user_features, item_features,
                     k_values=(5, 10, 20), num_threads=1):
    if test_interactions is None or test_interactions.nnz == 0:
        return {k: {"precision": 0., "recall": 0., "ndcg": 0., "hr": 0., "mrr": 0.}
                for k in k_values}

    test_csr  = test_interactions.tocsr()
    train_csr = train_interactions.tocsr()
    n_users, n_items = test_csr.shape
    max_k = max(k_values)

    accum = {k: {"precision": [], "recall": [], "ndcg": [], "hr": [], "mrr": []}
             for k in k_values}

    for u in range(n_users):
        true_items = set(test_csr[u].indices.tolist())
        if not true_items:
            continue

        scores = model.predict(
            user_ids=u, item_ids=np.arange(n_items),
            user_features=user_features, item_features=item_features,
            num_threads=num_threads,
        )

        train_items = train_csr[u].indices
        scores[train_items] = -np.inf

        top_idx = np.argpartition(-scores, max_k)[:max_k]
        top_idx = top_idx[np.argsort(-scores[top_idx])]

        relevance  = np.array([1.0 if i in true_items else 0.0 for i in top_idx])
        n_relevant = len(true_items)

        for k in k_values:
            rel_k  = relevance[:k]
            n_hits = float(rel_k.sum())

            precision = n_hits / k
            recall    = n_hits / n_relevant if n_relevant > 0 else 0.0
            hr        = 1.0 if n_hits > 0 else 0.0

            discounts = 1.0 / np.log2(np.arange(2, k + 2))
            dcg  = float((rel_k * discounts).sum())
            idcg = float(discounts[:min(n_relevant, k)].sum())
            ndcg = dcg / idcg if idcg > 0 else 0.0

            mrr = 0.0
            for j in range(k):
                if rel_k[j] > 0:
                    mrr = 1.0 / (j + 1)
                    break

            accum[k]["precision"].append(precision)
            accum[k]["recall"].append(recall)
            accum[k]["ndcg"].append(ndcg)
            accum[k]["hr"].append(hr)
            accum[k]["mrr"].append(mrr)

    return {
        k: {m: float(np.mean(v)) if v else 0.0 for m, v in metrics.items()}
        for k, metrics in accum.items()
    }


def print_metrics(metrics, label):
    print(f"\n=== {label} ===")
    print(f"{'K':>4} | {'Prec':>8} | {'Recall':>8} | {'NDCG':>8} | {'HR':>8} | {'MRR':>8}")
    print("-" * 58)
    for k in sorted(metrics.keys()):
        m = metrics[k]
        print(f"{k:>4} | {m['precision']:>8.4f} | {m['recall']:>8.4f} | "
              f"{m['ndcg']:>8.4f} | {m['hr']:>8.4f} | {m['mrr']:>8.4f}")

### 3.2 Huấn luyện LightFM với WARP
LightFM được huấn luyện theo từng epoch bằng WARP loss, với early stopping dựa trên NDCG@10 của tập validation. Bảng 1 là lịch sử huấn luyện được tạo bởi cell bên dưới.


In [ ]:
# [Giải thích cell]
# - Mục đích: huấn luyện LightFM bằng WARP trên dữ liệu train đã nhị phân hóa.
# - Đầu vào huấn luyện duy nhất: `train_interactions` cùng ma trận feature user/item.
# - Validation: `valid_inter` chỉ dùng để tính NDCG@10 và early stopping sau mỗi epoch.
# - Lưu ý: `test_inter` không tham gia huấn luyện hay chọn epoch.

model = LightFM(
    loss="warp",
    no_components=NO_COMPONENTS,
    learning_rate=LEARNING_RATE,
    item_alpha=ITEM_ALPHA,
    user_alpha=USER_ALPHA,
    random_state=42,
)

best_ndcg10      = -1.0
patience_counter = 0
history          = []

for epoch in range(1, MAX_EPOCHS + 1):
    model.fit_partial(
        interactions=train_interactions,
        user_features=user_feature_matrix,
        item_features=item_feature_matrix,
        epochs=1,
        num_threads=1,
        verbose=False,
    )

    valid_metrics = evaluate_metrics(
        model, valid_inter, train_interactions,
        user_feature_matrix, item_feature_matrix,
        k_values=K_VALUES, num_threads=NUM_THREADS,
    )

    ndcg10 = valid_metrics[10]["ndcg"]
    history.append({
        "epoch": epoch,
        **{f"ndcg@{k}": valid_metrics[k]["ndcg"] for k in K_VALUES}
    })
    print(f"Epoch {epoch:>2} | "
          + " | ".join(f"NDCG@{k}: {valid_metrics[k]['ndcg']:.4f}" for k in K_VALUES))

    if ndcg10 > best_ndcg10:
        best_ndcg10      = ndcg10
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

print(f"\n[PROPOSED-FINAL] Best Valid NDCG@10: {best_ndcg10:.4f}")

Epoch  1 | NDCG@5: 0.0068 | NDCG@10: 0.0093 | NDCG@20: 0.0125 | NDCG@50: 0.0190
Epoch  2 | NDCG@5: 0.0087 | NDCG@10: 0.0129 | NDCG@20: 0.0173 | NDCG@50: 0.0236
Epoch  3 | NDCG@5: 0.0110 | NDCG@10: 0.0153 | NDCG@20: 0.0205 | NDCG@50: 0.0286
Epoch  4 | NDCG@5: 0.0112 | NDCG@10: 0.0140 | NDCG@20: 0.0193 | NDCG@50: 0.0270
Epoch  5 | NDCG@5: 0.0086 | NDCG@10: 0.0111 | NDCG@20: 0.0191 | NDCG@50: 0.0238
Epoch  6 | NDCG@5: 0.0128 | NDCG@10: 0.0155 | NDCG@20: 0.0217 | NDCG@50: 0.0279
Epoch  7 | NDCG@5: 0.0121 | NDCG@10: 0.0174 | NDCG@20: 0.0186 | NDCG@50: 0.0233
Epoch  8 | NDCG@5: 0.0079 | NDCG@10: 0.0132 | NDCG@20: 0.0174 | NDCG@50: 0.0207
Epoch  9 | NDCG@5: 0.0135 | NDCG@10: 0.0147 | NDCG@20: 0.0202 | NDCG@50: 0.0266
Epoch 10 | NDCG@5: 0.0110 | NDCG@10: 0.0139 | NDCG@20: 0.0191 | NDCG@50: 0.0265
Epoch 11 | NDCG@5: 0.0151 | NDCG@10: 0.0176 | NDCG@20: 0.0238 | NDCG@50: 0.0270
Epoch 12 | NDCG@5: 0.0141 | NDCG@10: 0.0153 | NDCG@20: 0.0215 | NDCG@50: 0.0272
Epoch 13 | NDCG@5: 0.0123 | NDCG@10: 0.0

In [ ]:
# [Giải thích cell]
# - Mục đích: hiển thị lịch sử huấn luyện dưới dạng bảng để quan sát quá trình hội tụ.
# - Đầu vào chính: danh sách `history` được ghi lại sau mỗi epoch.
# - Kết quả tạo ra: `history_df`, bảng thể hiện loss/metric validation theo epoch.

history_df = pd.DataFrame(history)
display(history_df)

,epoch,ndcg@5,ndcg@10,ndcg@20,ndcg@50
0,1,0.006796,0.009304,0.012516,0.018997
1,2,0.008664,0.012852,0.017297,0.023614
2,3,0.011036,0.015319,0.020495,0.028627
3,4,0.011219,0.013997,0.019349,0.026960
4,5,0.008590,0.011099,0.019063,0.023758
5,6,0.012757,0.015534,0.021696,0.027930
6,7,0.012103,0.017420,0.018582,0.023306
7,8,0.007862,0.013179,0.017353,0.020667
8,9,0.013534,0.014739,0.020180,0.026629
9,10,0.011036,0.013909,0.019127,0.026549


### 3.3 Đánh giá cuối trên validation và test
Kết quả được báo cáo trên validation, test-full, test-warm và test-cold. Bảng 2 hỗ trợ so sánh hiệu năng tổng thể và khả năng xử lý item cold-start.


In [ ]:
# [Giải thích cell]
# - Mục đích: chạy đánh giá cuối cùng trên các split validation/test đã xác định.
# - Đầu vào chính: mô hình đã huấn luyện, train interactions và các tập held-out.
# - Kết quả tạo ra: bảng metric cho full, warm và cold split nếu có.
# - Lưu ý: không dùng kết quả test để chọn siêu tham số.

n_user_kg_tags = len({t for t in all_user_tags if t.startswith("user_kg_cluster:")})
n_tier3_tags   = len(all_user_tags) - n_user_kg_tags

print("=" * 70)
print("FINAL EVALUATION — KGE-UserNode")
print(f"Cấu hình : threshold={POSITIVE_THRESHOLD}, components={NO_COMPONENTS}")
print(f"Item   : {len(all_item_tags)} features (Tier1 + TextCluster + ItemKGECluster)")
print(f"User   : {len(all_user_tags)} features (Tier3={n_tier3_tags} + UserKGECluster={n_user_kg_tags})")
print("=" * 70)

for label, inter in [
    ("VALID",     valid_inter),
    ("TEST",      test_inter),
    ("TEST_WARM", test_warm_inter),
    ("TEST_COLD", test_cold_inter),
]:
    metrics = evaluate_metrics(
        model, inter, train_interactions,
        user_feature_matrix, item_feature_matrix,
        k_values=K_VALUES, num_threads=NUM_THREADS,
    )
    print_metrics(metrics, label)


FINAL EVALUATION — KGE-UserNode
Config : threshold=7, components=64
Item   : 940 features (Tier1 + TextCluster + ItemKGECluster)
User   : 49 features (Tier3=19 + UserKGECluster=30)

=== VALID ===
   K |     Prec |   Recall |     NDCG |       HR |      MRR
----------------------------------------------------------
   5 |   0.0033 |   0.0167 |   0.0107 |   0.0167 |   0.0087
  10 |   0.0025 |   0.0250 |   0.0133 |   0.0250 |   0.0097
  20 |   0.0019 |   0.0375 |   0.0164 |   0.0375 |   0.0105
  50 |   0.0014 |   0.0708 |   0.0231 |   0.0708 |   0.0117

=== TEST ===
   K |     Prec |   Recall |     NDCG |       HR |      MRR
----------------------------------------------------------
   5 |   0.0053 |   0.0267 |   0.0162 |   0.0267 |   0.0128
  10 |   0.0027 |   0.0267 |   0.0162 |   0.0267 |   0.0128
  20 |   0.0019 |   0.0382 |   0.0191 |   0.0382 |   0.0136
  50 |   0.0015 |   0.0725 |   0.0260 |   0.0725 |   0.0148

=== TEST_WARM ===
   K |     Prec |   Recall |     NDCG |       HR |   

### 3.4 Suy luận cho một người dùng
Cell này minh họa quy trình sinh top-$N$ gợi ý cho một người dùng cụ thể. Các item đã xuất hiện trong lịch sử train được loại bỏ khỏi danh sách ứng viên.


In [ ]:
# [Giải thích cell]
# - Mục đích: minh họa quy trình suy luận top-N cho một người dùng cụ thể.
# - Đầu vào chính: user_id, mô hình hoặc danh sách popularity, metadata phim và train history.
# - Kết quả tạo ra: bảng phim được đề xuất sau khi loại bỏ item người dùng đã thấy.
# - Lưu ý: đây là ví dụ định tính, không thay thế cho đánh giá định lượng Top-K.

def recommend_for_user(user_id, model, dataset, movies_df,
                       user_feature_matrix, item_feature_matrix,
                       train_df, n_recs=10):
    uid_map, _, iid_map, _ = dataset.mapping()
    if user_id not in uid_map:
        print(f"User {user_id} không tồn tại.")
        return pd.DataFrame()

    u_idx   = uid_map[user_id]
    n_items = item_feature_matrix.shape[0]

    scores = model.predict(
        user_ids=u_idx, item_ids=np.arange(n_items),
        user_features=user_feature_matrix, item_features=item_feature_matrix,
    )

    seen = set(train_df[train_df["id"] == user_id]["movie_id"].values)
    seen_indices = [iid_map[m] for m in seen if m in iid_map]
    scores[seen_indices] = -np.inf

    top_indices   = np.argsort(-scores)[:n_recs]
    idx2movie     = {v: k for k, v in iid_map.items()}
    top_movie_ids = [idx2movie[i] for i in top_indices]

    result = movies_df[movies_df["id"].isin(top_movie_ids)][
        ["id", "original_title", "genres", "year_published", "rate"]
    ].copy()
    result["predicted_score"] = [scores[iid_map[mid]] for mid in result["id"]]
    return result.sort_values("predicted_score", ascending=False)


sample_user = train_df["id"].iloc[0]
print(f"[PROPOSED-FINAL] Gợi ý top-10 cho user_id = {sample_user}\n")
recs = recommend_for_user(
    user_id=sample_user, model=model, dataset=dataset,
    movies_df=movies_df,
    user_feature_matrix=user_feature_matrix,
    item_feature_matrix=item_feature_matrix,
    train_df=train_df, n_recs=10,
)
print(recs.to_string(index=False))

[PROPOSED-FINAL] Gợi ý top-10 cho user_id = 103007

    id                      original_title                                          genres  year_published  rate  predicted_score
640188                    Band of Brothers                        Serie de TV|Bélico|Drama          2001.0   8.5      -384.208588
491709                             Amadeus                                           Drama          1984.0   7.7      -384.305603
847239  Sherlock: The Lying Detective (TV)                                   Intriga|Drama          2017.0   7.8      -384.491180
523268 Kokaku Kidotai (Ghost in the Shell)                Animación|Ciencia ficción|Acción          1995.0   7.5      -384.544067
173696                      Shutter Island                                Thriller|Intriga          2010.0   7.6      -384.632874
800220                             Aladdin Animación|Fantástico|Musical|Infantil|Aventuras          1992.0   7.4      -384.725952
315125                         Funny G

## 4. Kết luận (Conclusion)
Notebook này hoàn thiện quy trình thực nghiệm cho SeRel-LightFM đầy đủ. Các bước tiền xử lý, chia dữ liệu, huấn luyện và đánh giá được giữ thống nhất với baseline để kết quả có thể so sánh trực tiếp. Khi đặt cạnh các notebook baseline và ablation còn lại, kết quả của notebook này hỗ trợ phân tích đóng góp tương đối của metadata, embedding văn bản và embedding đồ thị tri thức đối với chất lượng gợi ý Top-$K$.
